# Unit 26 · Keras — First Contact & the Grammar Compared

**Learn with Adi — Python Programming (Bridge · Python for ML & DL)**

This is the notebook the unit page keeps pointing at. Keras and TensorFlow are far too large to run inside a browser, so every Keras example on the page is a read-only listing — **here they actually run**. Run a cell with **Shift+Enter**; the first Keras import takes a few seconds.

This is also the last notebook of the series. When you finish it, you are ready for the Machine Learning stage.

Study notes: https://aditya-402.github.io/learn-with-adi/series/python-programming/unit26.html

## 26.1 · Describe the model, don't assemble it

`Sequential([...])` is a **list** of layers. Around it sit three verbs: `compile` (settings), `fit` (show it examples), `predict` (ask about new data). Unit 25's five-line training loop lives inside `fit`.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

# the data: 4 students, 2 features each, one 0/1 answer each
X = np.array([[1.0, 7.0], [2.0, 6.0], [4.0, 5.0], [5.0, 7.0]])
y = np.array([0, 0, 1, 1])

# 1. DESCRIBE the model - a LIST of layers
model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

# 2. SET UP how it should learn
model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

# 3. SHOW it the examples, 200 times over
model.fit(X, y, epochs=200, verbose=0)

# 4. ASK: how did it do, and what about a new student?
loss, acc = model.evaluate(X, y, verbose=0)
print(f"loss {loss:.3f}  accuracy {acc:.2f}")
print(model.predict(np.array([[3.0, 6.0]]), verbose=0))

In [ ]:
# The summary Keras prints is exactly the parameter arithmetic from 26.1 P3:
# a Dense layer with n inputs and u units holds n*u weights + u biases.
model.summary()

In [ ]:
# 26.1 P3 - count the parameters yourself, in plain Python, then compare with summary() above.
units = [16, 8, 1]
n_in = 4
total = 0
# your loop here

## 26.2 · The three grammars, side by side

**Describe the model, show it examples, ask it about new data.** Three dialects, one sentence. Run the same tiny task two ways and watch only the ceremony change.

In [ ]:
# scikit-learn - the shortest dialect
import numpy as np
from sklearn.linear_model import LinearRegression

X = np.array([[1.0], [2.0], [3.0], [4.0]])   # 4 samples, 1 feature
y = np.array([3.0, 5.0, 7.0, 9.0])           # y = 2x + 1

model = LinearRegression()      # 1. describe the model
model.fit(X, y)                 # 2. show it examples
print(model.predict([[5.0]]))   # 3. ask about new data

In [ ]:
# Keras - same task, same three beats, plus a compile step
from tensorflow import keras
from tensorflow.keras import layers

kmodel = keras.Sequential([keras.Input(shape=(1,)), layers.Dense(1)])
kmodel.compile(optimizer="adam", loss="mse")
kmodel.fit(X, y, epochs=500, verbose=0)
print(kmodel.predict(np.array([[5.0]]), verbose=0))

# Probably not as accurate as the straight line, and it took 500 epochs to get there.
# Choosing a neural network for a straight-line problem is a choice, not an upgrade.

In [ ]:
# 26.2 P3 - fake the progress bar. fit() is a loop somebody else wrote.
losses = [0.9124, 0.5310, 0.3078]
# print "Epoch 1/3 - loss: 0.9124" and so on, then a closing line

## 26.3 · Which one, when

- **scikit-learn** — classic tabular work: rows, columns, a few thousand of them.
- **Keras** — a standard network, fast.
- **PyTorch** — when you want to see and control every step (and because research, and the whole LLM world, is written in it).

The *Build an LLM from Scratch* series on this site is PyTorch, in exactly Unit 25's vocabulary. Nothing further to learn before you start it.

In [ ]:
CHOICE = {
    "spreadsheet of customers": "scikit-learn",
    "image classifier, standard": "Keras",
    "a new idea from a paper": "PyTorch",
    "an LLM from scratch": "PyTorch",
}
jobs = ["spreadsheet of customers", "a new idea from a paper",
        "an LLM from scratch", "something nobody listed"]
for job in jobs:
    print(f"{job}: {CHOICE.get(job, 'read the docs, then pick')}")

## 26.4 · Reading ML code in the wild

Four questions, in order: **which library** (the imports) · **where does the data come in** · **where is the model described** · **where does learning happen** (`.fit(` or `for epoch in ...`). Skipping the unfamiliar lines is allowed — it's the professional move.

In [ ]:
snippet = """import torch
import torch.nn as nn
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
"""

if "torch" in snippet:
    print("Library: PyTorch")
elif "keras" in snippet or "tensorflow" in snippet:
    print("Library: Keras")
elif "sklearn" in snippet:
    print("Library: scikit-learn")
else:
    print("Library: not an ML file, or not one I know")

print("Note: nn.Sequential exists in PyTorch too - the import is the giveaway.")

In [ ]:
# 26.4 P3 - label every line by its role: data / describe / learn / ask / setup
program = [
    "import pandas as pd",
    "df = pd.read_csv('sales.csv')",
    "X = df[['size', 'rooms']].to_numpy()",
    "y = df['price'].to_numpy()",
    "model = LinearRegression()",
    "model.fit(X, y)",
    "print(model.predict(X[:2]))",
]
# your scanner here

## 26.5 · Your build — everything up to the model

Half (a) is the part these notes taught you properly: build the table, clean it, split into features and target, check the shapes, scale the columns. Half (b) hands the result to a small Keras model — and here, unlike on the page, it runs.

In [ ]:
import pandas as pd

# (a) everything up to the model - the part you can already do
df = pd.DataFrame({
    "hours":  [1.0, 2.0, None, 4.0, 5.0],
    "slept":  [7.0, 6.0, 8.0, 5.0, 7.0],
    "passed": [0, 0, 1, 1, 1],
})
print("rows before:", len(df))
df = df.dropna()
print("rows after:", len(df))

X = df[["hours", "slept"]].to_numpy()
y = df["passed"].to_numpy()
X = (X - X.mean(axis=0)) / X.std(axis=0)
print("X", X.shape, " y", y.shape)

In [ ]:
# (b) the same arrays, through a small Keras model - the next stage's subject
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history = model.fit(X, y, epochs=100, verbose=0)

loss, acc = model.evaluate(X, y, verbose=0)
print(f"loss {loss:.3f}  accuracy {acc:.2f}")

# An accuracy of 1.00 on four rows means nothing at all. Why that is,
# and what to do instead, is the first honest lesson of Machine Learning.

In [ ]:
# fit() hands back a history - the loss at every epoch. Plot it and you can SEE the loop.
import matplotlib.pyplot as plt

plt.plot(history.history["loss"])
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("what fit() was doing all along")
plt.show()

## Problem bank

1. **The dialect phrasebook** — `input()` a library name (`sklearn`, `keras`, `torch`) and print how it describes a model. Here `input()` genuinely waits for you.
2. **Shape check for a Dense layer** — 5 samples, 3 features, `Dense(2)`: build the zero-filled arrays, compute `X @ W + b`, print all four shapes and the parameter count.
3. **From a table to arrays** — turn a `city` column into a 0/1 `is_pune` feature, build `X` and `y`, print the shapes and the mean price.
4. **Stretch: build your own Sequential** — write `summary(layers, n_in)` over a list of layer dicts and print each layer's parameter count, the total, and the final output size. You are writing `model.summary()`.

Then, when you're done: open the Projects gallery and build something small and useless and yours. That's the whole point. — Adi

In [ ]:
# your problem-bank workspace
